<a href="https://colab.research.google.com/github/SriSharanya-617/encoder-decoder/blob/main/encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Generate output Word-by-Word

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Sample dataset
data = [
    ("i am learning", "je suis en train d apprendre"),
    ("he is running", "il court"),
    ("she is happy", "elle est heureuse"),
    ("i am happy", "je suis heureux")
]

input_texts = [x[0] for x in data]
target_texts = ["<start> " + x[1] + " <end>" for x in data]

tokenization

In [ ]:
# Encoder tokenizer
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

# Decoder tokenizer
target_tokenizer = Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)

padding

In [ ]:
max_input_len = max(len(seq) for seq in input_sequences)
max_target_len = max(len(seq) for seq in target_sequences)

encoder_input_data = pad_sequences(input_sequences, maxlen=max_input_len, padding='post')
decoder_input_data = pad_sequences(target_sequences, maxlen=max_target_len, padding='post')

create decoder target(shifted)

* decoder input:
* decoder output:





In [ ]:
decoder_target_data = np.zeros_like(decoder_input_data)

for i in range(len(data)):
    decoder_target_data[i, :-1] = decoder_input_data[i, 1:]

build model


paramters

In [ ]:
vocab_size_input = len(input_tokenizer.word_index) + 1
vocab_size_target = len(target_tokenizer.word_index) + 1
embedding_dim = 50
latent_dim = 100

encoder

In [ ]:
encoder_inputs=Input(shape=(None,))
enc_emb=Embedding(vocab_size_input,embedding_dim)(encoder_inputs)

encoder_lstm=LSTM(latent_dim,return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
#Encoder understands meaning and stores it in state_h, start_c, These togeather from Context vector
encoder_states = [state_h, state_c]
print(encoder_states)

[<KerasTensor shape=(None, 100), dtype=float32, sparse=False, ragged=False, name=keras_tensor_58>, <KerasTensor shape=(None, 100), dtype=float32, sparse=False, ragged=False, name=keras_tensor_59>]


In [ ]:
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(vocab_size_target, embedding_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_size_target, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

compile model

In [ ]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

Train model

In [ ]:
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=200
)

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - loss: 2.7070
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2.6852
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 2.6659
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2.6398
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2.6058
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2.5689
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 2.5130
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2.4220
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 2.3035
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 2.1416
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 1.9678
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.8042
Epoch 13/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.7790
Epoch 14/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 1.7876
Epoch 15/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 1.7636
Epoch 16/200
2/2 ━━

Inference Model

Training model # Prediction model
We build separate inference models

encoding interface

In [ ]:
encoder_model=Model(encoder_inputs,encoder_states)

Decoder interface

In [ ]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = Embedding(vocab_size_target, embedding_dim)(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)

decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

In [ ]:
reverse_target_index = {i: word for word, i in target_tokenizer.word_index.items()}

def decode_sequence(input_seq):

    states_value = encoder_model.predict(input_seq)

    start_token = target_tokenizer.word_index.get('<start>')

    if start_token is None:
        start_token = target_tokenizer.word_index.get('start')

    end_token = target_tokenizer.word_index.get('<end>')

    if end_token is None:
        end_token = target_tokenizer.word_index.get('end')

    target_seq = np.array([[start_token]])

    stop_condition = False
    decoded_sentence = ""

    while not stop_condition:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        sampled_word = reverse_target_index.get(
            sampled_token_index, ''
        )

        if (
            sampled_token_index == end_token or
            len(decoded_sentence.split()) > max_target_len
        ):
            stop_condition = True
        else:
            decoded_sentence += " " + sampled_word

        target_seq = np.array([[sampled_token_index]])

        states_value = [h, c]

    return decoded_sentence

test


In [ ]:
test_input = "i am happy"
seq = input_tokenizer.texts_to_sequences([test_input])
seq = pad_sequences(seq, maxlen=max_input_len, padding='post')

print("Input:", test_input)
print("Output:", decode_sequence(seq))

Input: i am happy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Output:  je suis heureux
